# Loss function modifications


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import multiprocessing
import matplotlib.pyplot as plt
import warnings
import sys
from scipy.optimize import curve_fit 

# --- Global Constants ---
N_RESTARTS = 5
OPTIM_STEPS_STAGE_1 = 1000
OPTIM_STEPS_STAGE_2 = 1000
OPTIM_STEPS_STAGE_3 = 1000
LEARNING_RATE = 0.0005
PHYSICAL_BOUNDARY_SECONDS = 1.25

# --- new constants ---
RARE_TRIAL_WEIGHT    = 2.5   # importance multiplier for rare/surprising trials
BARRIER_K            = 0.01  # barrier strength  (~1-2 % of typical NLL at init)
BARRIER_MARGIN       = 0.04  # fraction of param range where barrier activates
TRANSITION_LOSS_WEIGHT = 8.0 # weight for transition-asymmetry term in restart selection
N_POST_SWITCH        = 20    # trials after a block switch to include in transition curve
 

# ------------------------------------------------------------------
# 1.  DATA LOADING & CONTINGENCY PROCESSING
# ------------------------------------------------------------------
from Module.Reader import load_mat
from Module.session_name_parse import parse_behavior_file_path
from Module.Licking_properties import extract_lick_properties

def states_labeling(trial_states):
    outcome_priority = [
        'ChangingMindReward', 'WrongInitiation', 'Punish', 'Reward',
        'PunishNaive', 'RewardNaive', 'EarlyChoice', 'DidNotChoose'
    ]
    for state in outcome_priority:
        if (state in trial_states and 
            len(trial_states[state]) > 0 and 
            not np.isnan(trial_states[state][0])):
            return state, trial_states[state][0]
    return 'Other', np.nan

def extract_session_properties(data_single_session, subject, version, session_date):
    try:
        settings = data_single_session['TrialSettings'][0]['GUI']
        gui_meta = data_single_session.get('TrialSettings', [{}])[0].get('GUIMeta', {})
        
        number_of_trials = data_single_session.get('nTrials', 0)
        warm_up_trials = settings.get('NumNaiveWarmup', 0)
        max_same_side = settings.get('MaxSameSide', 0)
        Num_block_warmup = settings.get('NumBlockWarmup', 0)
        block_length = settings.get('BlockLength', 0)
        
        training_level_idx = settings.get('TrainingLevel', 0)
        training_level = gui_meta.get('TrainingLevel', {}).get('String', [])[training_level_idx-1] if 'TrainingLevel' in gui_meta else 'Unknown'
        
        if 'Contingency' in settings:
            contingency_idx = settings.get('Contingency', 0)
            contingency_type = gui_meta.get('Contingency', {}).get('String', [])[contingency_idx - 1] if 'Contingency' in gui_meta else 'Unknown'
        else:
            contingency_type = 'Normal - Short/Left'
            contingency_idx = 1 

        isi_settings = {
            'short': {'mean': settings.get('ISIShortMean_s', 0), 'max': settings.get('ISIShortMax_s', 0), 'min': settings.get('ISIShortMin_s', 0)},
            'long': {'mean': settings.get('ISILongMean_s', 0), 'max': settings.get('ISILongMax_s', 0), 'min': settings.get('ISILongMin_s', 0)}
        }
        isi_divider = settings.get('ISIOrig_s', 0)
        
        trials_properties = {
            'subject': subject, 'version': version, 'session_date': session_date,
            'outcome': [], 'outcome_initiate': [], 'warm_up': [], 'trial_initiation_time': [],
            'jitter_flag': [], 'opto_tag': [], 'trial_type': [], 'trial_isi': [], 'block_type': [],  
            'n_warm_up_trials': warm_up_trials, 'n_max_same_side': max_same_side,
            'difficulty': training_level, 'experimentor': settings.get('ExperimenterInitials', 'Unknown'),
            'Anti_bias': settings.get('AntiBiasServoAdjustAct', 0), 'num_warmup_blocks': Num_block_warmup,
            'block_length': block_length, 'Contingency': contingency_type, 'Contingency_idx': contingency_idx,
            'isi_settings': isi_settings, 'isi_devider': isi_divider
        }

        trial_types = data_single_session.get('TrialTypes', [])
        block_types = data_single_session.get('BlockTypes', [])
        
        for i in range(number_of_trials):
            try:
                trial_states = data_single_session['RawEvents']['Trial'][i]['States']
                outcome, outcome_initiate = states_labeling(trial_states)
                trials_properties['outcome'].append(outcome)
                trials_properties['outcome_initiate'].append(outcome_initiate)
                trials_properties['warm_up'].append(1 if i < warm_up_trials else 0)
                trials_properties['trial_initiation_time'].append(outcome_initiate)
                
                jitter_flag = data_single_session.get('JitterFlag', [])
                trials_properties['jitter_flag'].append(1 if i < len(jitter_flag) and jitter_flag[i] == 1 else 0)
                
                opto_type = data_single_session.get('OptoType', [])
                trials_properties['opto_tag'].append(opto_type[i] if i < len(opto_type) else None)
                
                trial_type = None
                if i < len(trial_types):
                    if contingency_idx == 1:
                        trial_type = trial_types[i]
                    elif contingency_idx == 2:
                        trial_type = 2 if trial_types[i] == 1 else (1 if trial_types[i] == 2 else trial_types[i])
                    else:
                        trial_type = trial_types[i]
                trials_properties['trial_type'].append(trial_type)

                block_type = None
                if i < len(block_types):
                    if contingency_idx == 1:
                        block_type = block_types[i]
                    elif contingency_idx == 2:
                        block_type = 2 if block_types[i] == 1 else (1 if block_types[i] == 2 else block_types[i])
                    else:
                        block_type = block_types[i]
                trials_properties['block_type'].append(block_type)
                
                processed_data = data_single_session.get('ProcessedSessionData', [])
                if i < len(processed_data) and isinstance(processed_data[i], dict) and 'trial_isi' in processed_data[i]:
                    trials_properties['trial_isi'].append(processed_data[i]['trial_isi'].get('PostMeanISI', None))
                else:
                    trials_properties['trial_isi'].append(None)
                    
            except (KeyError, IndexError):
                for key in ['outcome', 'outcome_initiate', 'warm_up', 'trial_initiation_time', 
                           'jitter_flag', 'opto_tag', 'trial_type', 'trial_isi', 'block_type']:
                    if len(trials_properties[key]) == i:
                        trials_properties[key].append(None)
        
        return trials_properties
    except Exception as e:
        print(f"Error extracting session properties: {e}")
        return {}

def load_session_data(path):
    session = load_mat(path)
    subject, version, date = parse_behavior_file_path(path)
    return {
        'session': session, 'subject': subject, 'version': version, 'date': date,
        'trial_properties': extract_session_properties(session, subject, version, date),
        'lick_properties': extract_lick_properties(session, subject, version, date)
    }

def prepare_session_data(data_paths):
    sessions_data = {
        'dates': [], 'outcomes': [], 'opto_tags': [], 'trial_types': [], 'trial_isi': [],
        'lick_properties': [], 'block_type': []
    }
    for path in data_paths:
        try:
            data = load_session_data(path)
            sessions_data['dates'].append(data['date'])
            sessions_data['outcomes'].append(data['trial_properties']['outcome'])
            sessions_data['opto_tags'].append(data['trial_properties']['opto_tag'])
            sessions_data['trial_types'].append(data['trial_properties']['trial_type'])
            sessions_data['trial_isi'].append(data['trial_properties']['trial_isi'])
            sessions_data['lick_properties'].append(data['lick_properties'])
            sessions_data['block_type'].append(data['trial_properties']['block_type'])
        except Exception as e:
            print(f"Warning: Could not load {path}: {e}")
    return sessions_data

def transform_data_to_dataframe(sessions_data, session_index=0):
    if session_index >= len(sessions_data['outcomes']): return pd.DataFrame()
    outcomes = np.asarray(sessions_data['outcomes'][session_index])
    trial_types_int = np.asarray(sessions_data['trial_types'][session_index])
    trial_isi = np.asarray(sessions_data['trial_isi'][session_index])
    block_types = np.asarray(sessions_data['block_type'][session_index])

    if len(outcomes) == 0: return pd.DataFrame()

    mouse_choices = []
    for outcome, tt in zip(outcomes, trial_types_int):
        rewarded = outcome in ['Reward', 'RewardNaive']
        is_short = (tt == 1)
        if rewarded:
            mouse_choices.append('left' if is_short else 'right')
        else:
            mouse_choices.append('right' if is_short else 'left')

    df = pd.DataFrame({
        'isi': trial_isi,
        'trial_type': ['short' if tt == 1 else 'long' for tt in trial_types_int],
        'block_type': ['neutral' if b == 0 else 'short_block' if b == 1 else 'long_block' for b in block_types],
        'mouse_choice': mouse_choices,
        'rewarded': [1 if o in ['Reward', 'RewardNaive'] else 0 for o in outcomes]
    })
    return df.dropna().reset_index(drop=True)

# ------------------------------------------------------------------
# 2.  MOUSE MODEL CLASS (14 Parameters)
# ------------------------------------------------------------------
class MouseModelTorch(nn.Module):
    def __init__(self,
                 decay_rate, noise_param_a, 
                 alpha_reward_base, alpha_punish_base, 
                 gamma, alpha_unc_ctx, alpha_unc_sens,
                 alpha_choice_reward, alpha_choice_punish,
                 beta, lapse_rate, subjective_p_switch, subjective_p_rare, 
                 alpha_boundary, 
                 fixed_bias_value=0.0, initial_boundary=PHYSICAL_BOUNDARY_SECONDS): 
        super().__init__()
        
        self.decay_rate = decay_rate
        self.noise_param_a = noise_param_a
        self.alpha_reward_base = alpha_reward_base
        self.alpha_punish_base = alpha_punish_base
        self.gamma = torch.tensor(gamma, dtype=torch.float32) 
        self.alpha_unc_ctx = alpha_unc_ctx
        self.alpha_unc_sens = alpha_unc_sens
        self.alpha_choice_reward = alpha_choice_reward
        self.alpha_choice_punish = alpha_choice_punish  
        self.beta = beta
        self.lapse = lapse_rate
        
        self.p_switch = torch.tensor(subjective_p_switch, dtype=torch.float32)
        self.p_rare = torch.tensor(subjective_p_rare, dtype=torch.float32)
        self.p_common = 1.0 - self.p_rare
        self.alpha_boundary = torch.tensor(alpha_boundary, dtype=torch.float32)
        self.normalizer = torch.tensor(1.0, dtype=torch.float32)
        self.static_bias = torch.tensor(fixed_bias_value, dtype=torch.float32)
        self.initial_boundary = torch.tensor(initial_boundary, dtype=torch.float32)
        
        self.reset_state()

    def reset_state(self):
        self.weights = torch.tensor([0.5, 0.5], dtype=torch.float32)
        self.p_short_block = torch.tensor(0.5, dtype=torch.float32)
        self.last_choice_bias = torch.tensor(0.0, dtype=torch.float32)
        self.current_boundary = self.initial_boundary.clone()

    @torch.jit.export
    def _time_perception_reparam(self, isi_expanded):
        lam = self.decay_rate
        isi_safe = torch.clamp(isi_expanded, min=0.0)
        cdf_val = 1.0 - (1.0 + lam * isi_safe) * torch.exp(-lam * isi_safe)
        mean_perc = self.normalizer * cdf_val
        std_perc = mean_perc / self.noise_param_a
        eps = torch.randn_like(mean_perc)
        perceived_time = mean_perc + eps * torch.clamp(std_perc, min=1e-6)
        return torch.clamp(perceived_time, min=0.01)

    @torch.jit.export
    def _update_context_belief(self, is_short_trial):
        p_short_prior = self.p_short_block * (1 - self.p_switch) + (1 - self.p_short_block) * self.p_switch
        L_short = torch.where(is_short_trial, self.p_common, self.p_rare)
        L_long = torch.where(is_short_trial, self.p_rare, self.p_common)
        num = L_short * p_short_prior
        den = num + L_long * (1 - p_short_prior)
        self.p_short_block = num / (den + 1e-12)

    @torch.jit.export
    def get_choice_probabilities(self, isi: torch.Tensor, current_phys_bound: torch.Tensor, n_samples: int = 50):
        cdf_bound = 1.0 - (1.0 + self.decay_rate * current_phys_bound) * torch.exp(-self.decay_rate * current_phys_bound)
        internal_boundary = self.normalizer * cdf_bound

        isi_expanded = isi.expand(n_samples) 
        perceived_time_vec = self._time_perception_reparam(isi_expanded)
        
        dist = perceived_time_vec - internal_boundary
        exp_arg = torch.clamp(-dist, -50.0, 50.0) 
        sigmoid_val = 1.0 / (1.0 + torch.exp(exp_arg))
        time_evidence = (2.0 * sigmoid_val - 1.0) 
        
        context_evidence = (0.5 - self.p_short_block) 
        dv_time = self.weights[0] * time_evidence
        dv_ctx = self.weights[1] * context_evidence
        decision_variable = dv_time + dv_ctx + self.last_choice_bias + self.static_bias
        
        dv_scaled = torch.clamp(self.beta * decision_variable, -50.0, 50.0)
        prob_left_no_lapse = 1.0 / (1.0 + torch.exp(dv_scaled))
        
        avg_prob = torch.mean(prob_left_no_lapse)
        prob_left = avg_prob * (1 - self.lapse) + 0.5 * self.lapse
        
        return prob_left, torch.mean(perceived_time_vec), internal_boundary

    @torch.jit.export
    def update_weights(self, was_correct, is_short_trial, perceived_time, choice_was_left):
        cdf_bound = 1.0 - (1.0 + self.decay_rate * self.current_boundary) * torch.exp(-self.decay_rate * self.current_boundary)
        internal_boundary = self.normalizer * cdf_bound

        unc = 4.0 * self.p_short_block * (1.0 - self.p_short_block)
        was_correct_f = was_correct.float()
        
        dist = perceived_time - internal_boundary
        exp_arg = torch.clamp(-dist, -50.0, 50.0)
        sigmoid_val = 1.0 / (1.0 + torch.exp(exp_arg))
        c_sensory = torch.abs(2.0 * sigmoid_val - 1.0)

        alpha_base = was_correct_f * self.alpha_reward_base + (1 - was_correct_f) * self.alpha_punish_base
        alpha_ctx = alpha_base * (1.0 + self.alpha_unc_ctx * unc) * self.gamma
        alpha_sensory = alpha_base * (1.0 + self.alpha_unc_sens * c_sensory) * (1.0 - self.gamma)
        
        is_perc_short = (perceived_time < internal_boundary)
        time_ok_bool = (is_perc_short == is_short_trial)
        context_evidence = 0.5 - self.p_short_block
        
        dir_time = 2.0 * time_ok_bool.float() - 1.0
        if context_evidence > 0:
            dir_ctx = torch.where(is_short_trial, torch.tensor(1.0), torch.tensor(-1.0))
        elif context_evidence < 0:
            dir_ctx = torch.where(is_short_trial, torch.tensor(-1.0), torch.tensor(1.0))
        else:
            dir_ctx = torch.tensor(0.0)
            
        w_time_new = self.weights[0] + alpha_sensory * dir_time
        w_ctx_new = self.weights[1] + alpha_ctx * dir_ctx
        self.weights = torch.clamp(torch.stack([w_time_new, w_ctx_new]), min=0.0)
        
        target = torch.where(choice_was_left, torch.tensor(-1.0), torch.tensor(1.0))
        a_hist = torch.where(was_correct, self.alpha_choice_reward, self.alpha_choice_punish)
        target_bias = torch.where(was_correct, target, -target)
        self.last_choice_bias = self.last_choice_bias + a_hist * (target_bias - self.last_choice_bias)
        self._update_context_belief(is_short_trial)

# ------------------------------------------------------------------
# 3.  FITTING LOGIC
# ------------------------------------------------------------------

# ------------------------------------------------------------------
# NEW HELPER 1 — soft barrier penalty
# Called from fit_single_session_torch BEFORE the JIT call so that
# gradients flow through params_vec cleanly.
# ------------------------------------------------------------------
def compute_soft_barrier_penalty(
    params_vec: torch.Tensor,
    bounds: list,
    k: float = BARRIER_K,
    margin: float = BARRIER_MARGIN
) -> torch.Tensor:
    """
    Returns a scalar penalty that is near-zero when all parameters are
    in the interior of their bounds, and rises sharply as any parameter
    approaches a hard bound.
 
    params_vec : 1-D tensor of length len(bounds), requires_grad=True
    bounds     : list of (lo, hi) tuples — same order as params_vec
    k          : barrier strength scalar
    margin     : fraction of (hi-lo) that defines the 'danger zone'
 
    The barrier activates only inside the margin region near each wall:
        lo_inner = lo + margin*(hi-lo)
        hi_inner = hi - margin*(hi-lo)
    Outside [lo_inner, hi_inner] a 1/distance term is added.
    Inside that region the penalty is exactly zero.
 
    Anchoring with `params_vec.sum() * 0.0` keeps the computation
    graph connected to params_vec so autograd works even when all
    parameters are in the interior (penalty = 0 but grad_fn exists).
    """
    # differentiable zero — keeps the graph connected to params_vec
    penalty = params_vec.sum() * 0.0
    eps = 1e-6
 
    for i, (lo, hi) in enumerate(bounds):
        p        = params_vec[i]
        rng      = hi - lo
        lo_inner = lo + margin * rng
        hi_inner = hi - margin * rng
 
        dist_lo = p - lo + eps
        dist_hi = hi - p + eps
 
        # use .float() mask so gradient flows through the 1/dist terms
        in_lo = (p < lo_inner).float()
        in_hi = (p > hi_inner).float()
 
        penalty = penalty + in_lo * k / dist_lo + in_hi * k / dist_hi
 
    return penalty
 
 
# ------------------------------------------------------------------
# NEW HELPER 2 — transition asymmetry loss  (pure numpy, post-hoc)
# Called once per restart on the final sim_df to rank restarts.
# ------------------------------------------------------------------
def compute_transition_asymmetry_loss(
    sim_df: pd.DataFrame,
    real_df: pd.DataFrame,
    n_post_switch: int = N_POST_SWITCH
) -> float:
    """
    Computes the mean squared error between the model's post-switch
    accuracy curves and the real mouse's post-switch accuracy curves,
    separately for short→long and long→short transitions.
 
    sim_df   : output of simulate_detailed_session_torch — must contain
               'block_type' and 'correct_model' columns.
    real_df  : the session dataframe — must contain
               'block_type' and 'rewarded' columns.
    Returns a float scalar (MSE, lower = better).
    """
 
    def _transition_curves(df, correct_col):
        """
        Returns dict:
            {'short_to_long': array[n_post_switch],
             'long_to_short': array[n_post_switch]}
        Each array is mean accuracy at positions 1..n_post_switch
        after the respective switch type. NaN where no data.
        """
        blocks = df['block_type'].values
        correct = df[correct_col].values.astype(float)
 
        curves = {
            'short_to_long': np.full(n_post_switch, np.nan),
            'long_to_short': np.full(n_post_switch, np.nan),
        }
        accum = {
            'short_to_long': [[] for _ in range(n_post_switch)],
            'long_to_short': [[] for _ in range(n_post_switch)],
        }
 
        n = len(blocks)
        for t in range(1, n):
            prev, curr = blocks[t - 1], blocks[t]
            # detect transition type
            if prev == 'short_block' and curr == 'long_block':
                key = 'short_to_long'
            elif prev == 'long_block' and curr == 'short_block':
                key = 'long_to_short'
            else:
                continue  # not a switch or involves neutral block
 
            # collect the next n_post_switch trials
            for offset in range(n_post_switch):
                idx = t + offset
                if idx >= n:
                    break
                # stop collecting if another switch occurs
                if idx > t and blocks[idx] != curr:
                    break
                accum[key][offset].append(correct[idx])
 
        for key in curves:
            for pos in range(n_post_switch):
                vals = accum[key][pos]
                if len(vals) >= 2:            # need at least 2 observations
                    curves[key][pos] = np.mean(vals)
 
        return curves
 
    # model correctness column differs from real data column
    # sim_df uses 'correct_model' (bool), real_df uses 'rewarded' (0/1)
    sim_curves  = _transition_curves(sim_df,  'correct_model')
    real_curves = _transition_curves(real_df, 'rewarded')
 
    mse_total = 0.0
    n_valid   = 0
 
    for key in ['short_to_long', 'long_to_short']:
        s = sim_curves[key]
        r = real_curves[key]
        mask = ~np.isnan(s) & ~np.isnan(r)
        if mask.sum() > 0:
            mse_total += np.mean((s[mask] - r[mask]) ** 2)
            n_valid   += 1
 
    if n_valid == 0:
        return 0.0          # no switch data — don't penalise
    return mse_total / n_valid
 
 
# ------------------------------------------------------------------
# NEW HELPER 3 — composite restart selection
# ------------------------------------------------------------------
def select_best_restart(
    restart_results: list,
    real_df: pd.DataFrame,
    w_transition: float = TRANSITION_LOSS_WEIGHT
) -> dict:
    """
    Picks the best restart using:
        composite = nll  +  w_transition * transition_loss_normalised
 
    transition_loss is normalised to be in the same units as NLL by
    multiplying it by n_trials / (2 * N_POST_SWITCH), i.e. scaling it
    to be comparable across sessions of different lengths.
 
    restart_results : list of dicts returned by fit_single_session_torch
                      (each must have 'nll', 'sim_df', 'best_params',
                       'fixed_bias', 'success', 'loss_history')
    real_df         : the original session dataframe
    Returns the single best result dict.
    """
    n_trials = len(real_df)
    normaliser = n_trials / (2.0 * N_POST_SWITCH + 1e-6)
 
    best_score  = float('inf')
    best_result = restart_results[0]
 
    for res in restart_results:
        if not res['success']:
            continue
        t_loss = compute_transition_asymmetry_loss(res['sim_df'], real_df)
        composite = res['nll'] + w_transition * t_loss * normaliser
        res['transition_loss']  = t_loss
        res['composite_score']  = composite
        if composite < best_score:
            best_score  = composite
            best_result = res
 
    return best_result

def calculate_neutral_bias(df_session):
    neutral_df = df_session[df_session['block_type'] == 'neutral']
    if len(neutral_df) < 10: return 0.0
    p_left = (neutral_df['mouse_choice'] == 'left').mean()
    p_left = np.clip(p_left, 0.01, 0.99)
    return float(-np.log(p_left / (1.0 - p_left)))

def get_session_tensors(df_session):
    isi = torch.tensor(df_session['isi'].values, dtype=torch.float32)
    is_short = torch.tensor((df_session['trial_type'] == 'short').values, dtype=torch.bool)
    mouse_ch_left = torch.tensor((df_session['mouse_choice'] == 'left').values, dtype=torch.bool)
    return isi, is_short, mouse_ch_left

@torch.jit.script
def run_session_jit(
    decay: torch.Tensor, noise: torch.Tensor,
    a_rew: torch.Tensor, a_pun: torch.Tensor,
    gamma: torch.Tensor,
    a_unc_ctx: torch.Tensor, a_unc_sens: torch.Tensor,
    a_chr: torch.Tensor, a_chp: torch.Tensor,
    beta: torch.Tensor, lapse: torch.Tensor,
    p_switch: torch.Tensor, p_rare: torch.Tensor,
    alpha_bound: torch.Tensor,
    fixed_bias: torch.Tensor, start_phys_bound: torch.Tensor,
    isi_ten: torch.Tensor, is_short_ten: torch.Tensor,
    ch_left_ten: torch.Tensor,
    n_samples: int,
    rare_weight: float,         # NEW — RARE_TRIAL_WEIGHT constant
    barrier_penalty: torch.Tensor  # NEW — pre-computed barrier scalar
) -> torch.Tensor:
    """
    Returns weighted NLL + barrier_penalty.
 
    rare_weight    : scalar > 1.0; multiplies NLL for trials that are
                     surprising given current context belief.
    barrier_penalty: computed externally by compute_soft_barrier_penalty
                     and passed in so autograd traces through params_vec.
    """
 
    normalizer = 1.0
    p_common   = 1.0 - p_rare
 
    w_time       = torch.tensor(0.5)
    w_ctx        = torch.tensor(0.5)
    p_short      = torch.tensor(0.5)
    last_bias    = torch.tensor(0.0)
    curr_phys_bound = start_phys_bound
    nll_total    = torch.tensor(0.0)
    eps_log      = 1e-9
 
    N = isi_ten.size(0)
 
    for i in range(N):
        isi_t     = isi_ten[i]
        is_short_t = is_short_ten[i]
        ch_left_t  = ch_left_ten[i]
 
        # ---- boundary in perceptual space ----
        cdf_bound     = 1.0 - (1.0 + decay * curr_phys_bound) * torch.exp(-decay * curr_phys_bound)
        internal_boundary = normalizer * cdf_bound
 
        # ---- perceptual samples ----
        isi_exp   = isi_t.expand(n_samples)
        isi_safe  = torch.clamp(isi_exp, min=0.0)
        cdf_val   = 1.0 - (1.0 + decay * isi_safe) * torch.exp(-decay * isi_safe)
        mean_perc = normalizer * cdf_val
        std_perc  = mean_perc / noise
        noise_vec = torch.randn_like(mean_perc)
        perceived_time_vec = mean_perc + noise_vec * torch.clamp(std_perc, min=1e-6)
        perceived_time_vec = torch.clamp(perceived_time_vec, min=0.01)
 
        # ---- decision variable ----
        dist      = perceived_time_vec - internal_boundary
        exp_arg   = torch.clamp(-dist, -50.0, 50.0)
        sigmoid_val = 1.0 / (1.0 + torch.exp(exp_arg))
        time_evidence = 2.0 * sigmoid_val - 1.0
 
        context_evidence = 0.5 - p_short
        dv   = w_time * time_evidence + w_ctx * context_evidence + last_bias + fixed_bias
        dv_s = torch.clamp(beta * dv, -50.0, 50.0)
        p_left_vec = 1.0 / (1.0 + torch.exp(dv_s))
 
        avg_p  = torch.mean(p_left_vec)
        p_left = avg_p * (1.0 - lapse) + 0.5 * lapse
 
        # ---- NEW: per-trial importance weight ----
        # A trial is 'rare' when its type disagrees with the model's
        # current majority-block belief.
        # is_short_t=True  AND p_short < 0.5  → rare long block, short trial
        # is_short_t=False AND p_short >= 0.5 → rare short block, long trial
        trial_matches_belief = (is_short_t == (p_short >= 0.5))
        trial_weight = 1.0 if trial_matches_belief else rare_weight
 
        # ---- weighted NLL accumulation ----
        if ch_left_t:
            nll_total = nll_total - trial_weight * torch.log(p_left + eps_log)
        else:
            nll_total = nll_total - trial_weight * torch.log(1.0 - p_left + eps_log)
 
        # ---- state updates (identical to original) ----
        was_correct = (ch_left_t == is_short_t)
 
        bound_shift = torch.tensor(0.0)
        if not was_correct:
            if is_short_t:
                bound_shift = alpha_bound
            else:
                bound_shift = -alpha_bound
        curr_phys_bound = torch.clamp(curr_phys_bound + bound_shift, 0.2, 2.3)
 
        unc          = 4.0 * p_short * (1.0 - p_short)
        was_correct_f = float(was_correct)
 
        t_accum       = torch.mean(perceived_time_vec)
        dist_avg      = t_accum - internal_boundary
        exp_arg_avg   = torch.clamp(-dist_avg, -50.0, 50.0)
        sigmoid_avg   = 1.0 / (1.0 + torch.exp(exp_arg_avg))
        c_sensory     = torch.abs(2.0 * sigmoid_avg - 1.0)
 
        alpha_base    = was_correct_f * a_rew + (1.0 - was_correct_f) * a_pun
        alpha_ctx     = alpha_base * (1.0 + a_unc_ctx * unc) * gamma
        alpha_sensory = alpha_base * (1.0 + a_unc_sens * c_sensory) * (1.0 - gamma)
 
        is_perc_short = (t_accum < internal_boundary)
        time_ok       = (is_perc_short == is_short_t)
        dir_time      = 2.0 * float(time_ok) - 1.0
 
        context_evidence_upd = 0.5 - p_short
        if context_evidence_upd > 0:
            dir_ctx = 1.0 if is_short_t else -1.0
        elif context_evidence_upd < 0:
            dir_ctx = -1.0 if is_short_t else 1.0
        else:
            dir_ctx = 0.0
 
        w_time = w_time + alpha_sensory * dir_time
        w_ctx  = w_ctx  + alpha_ctx     * dir_ctx
        if w_time < 0.0: w_time = torch.tensor(0.0)
        if w_ctx  < 0.0: w_ctx  = torch.tensor(0.0)
 
        target      = -1.0 if ch_left_t else 1.0
        a_hist      = a_chr if was_correct else a_chp
        target_bias = target if was_correct else -target
        last_bias   = last_bias + a_hist * (target_bias - last_bias)
 
        p_short_prior = p_short * (1.0 - p_switch) + (1.0 - p_short) * p_switch
        if is_short_t:
            L_short, L_long = p_common, p_rare
        else:
            L_short, L_long = p_rare, p_common
        num     = L_short * p_short_prior
        den     = num + L_long * (1.0 - p_short_prior)
        p_short = num / (den + 1e-12)
 
    # ---- NEW: add pre-computed barrier penalty ----
    return nll_total + barrier_penalty

def fit_single_session_torch(args):
    session_data, session_index, bounds, _ = args
 
    fixed_bias      = torch.tensor(calculate_neutral_bias(session_data), dtype=torch.float32)
    isi_ten, is_short_ten, ch_left_ten = get_session_tensors(session_data)
    start_phys_bound = torch.tensor(PHYSICAL_BOUNDARY_SECONDS, dtype=torch.float32)
 
    low_b  = torch.tensor([b[0] for b in bounds], dtype=torch.float32)
    high_b = torch.tensor([b[1] for b in bounds], dtype=torch.float32)
 
    # index groups that match the parameter ORDER in bounds / params_vec:
    # 0:decay  1:noise  2:a_rew  3:a_pun  4:gamma
    # 5:a_unc_ctx  6:a_unc_sens  7:a_chr  8:a_chp
    # 9:beta  10:lapse  11:p_switch  12:p_rare  13:alpha_bound
    PERCEPTUAL_IDX = [0, 1, 9, 10]       # decay, noise, beta, lapse
    LEARNING_IDX   = [2, 3, 4, 5, 6, 7, 8, 11, 12, 13]   # everything else
 
    all_restart_results = []
    restart_pbar = tqdm(range(N_RESTARTS), desc=f"Sess {session_index} Restarts", leave=False)
 
    for run_i in restart_pbar:
        torch.manual_seed(session_index * 1000 + run_i)
        current_run_history = []
 
        init_vals  = [(torch.rand(1) * (hi - lo) + lo).item() for lo, hi in bounds]
        params_vec = torch.tensor(init_vals, dtype=torch.float32, requires_grad=True)
 
        def clamp_params():
            with torch.no_grad():
                params_vec.data = torch.max(torch.min(params_vec.data, high_b), low_b)
 
        def call_jit_model(p_vec, n_samp):
            # compute barrier each call so gradients flow through p_vec
            barrier = compute_soft_barrier_penalty(p_vec, bounds)
            return run_session_jit(
                decay=p_vec[0],  noise=p_vec[1],
                a_rew=p_vec[2],  a_pun=p_vec[3],
                gamma=p_vec[4],
                a_unc_ctx=p_vec[5], a_unc_sens=p_vec[6],
                a_chr=p_vec[7],  a_chp=p_vec[8],
                beta=p_vec[9],   lapse=p_vec[10],
                p_switch=p_vec[11], p_rare=p_vec[12],
                alpha_bound=p_vec[13],
                fixed_bias=fixed_bias,
                start_phys_bound=start_phys_bound,
                isi_ten=isi_ten,
                is_short_ten=is_short_ten,
                ch_left_ten=ch_left_ten,
                n_samples=n_samp,
                rare_weight=RARE_TRIAL_WEIGHT,    # NEW
                barrier_penalty=barrier           # NEW
            )
 
        # ----------------------------------------------------------------
        # STAGE 1 — perceptual params lead, learning params follow slowly
        # No zeroing. Differential learning rates instead.
        # ----------------------------------------------------------------
        stage1_groups = [
            {'params': [params_vec], 'lr': LEARNING_RATE,        # full rate for all
             'param_indices': list(range(len(bounds)))}           # but we override below
        ]
        # PyTorch doesn't support per-element LR in a single tensor, so we
        # implement differential LR via gradient masking after each step:
        # multiply gradients of slow-group params by 0.05 before the update.
 
        optimizer_s1 = optim.Adam([params_vec], lr=LEARNING_RATE)
 
        for step in range(OPTIM_STEPS_STAGE_1):
            restart_pbar.set_postfix({'Stage': '1/3', 'Step': step + 1})
            optimizer_s1.zero_grad()
            loss = call_jit_model(params_vec, n_samp=50)
            loss.backward()
 
            # slow down learning-param gradients in stage 1
            with torch.no_grad():
                for idx in LEARNING_IDX:
                    if params_vec.grad is not None:
                        params_vec.grad[idx] *= 0.05
 
            optimizer_s1.step()
            clamp_params()
            current_run_history.append(loss.item())
 
        # ----------------------------------------------------------------
        # STAGE 2 — learning params lead, perceptual params follow slowly
        # ----------------------------------------------------------------
        optimizer_s2 = optim.Adam([params_vec], lr=LEARNING_RATE)
 
        for step in range(OPTIM_STEPS_STAGE_2):
            restart_pbar.set_postfix({'Stage': '2/3', 'Step': step + 1})
            optimizer_s2.zero_grad()
            loss = call_jit_model(params_vec, n_samp=30)
            loss.backward()
 
            # slow down perceptual-param gradients in stage 2
            with torch.no_grad():
                for idx in PERCEPTUAL_IDX:
                    if params_vec.grad is not None:
                        params_vec.grad[idx] *= 0.05
 
            optimizer_s2.step()
            clamp_params()
            current_run_history.append(loss.item())
 
        # ----------------------------------------------------------------
        # STAGE 3 — full joint optimisation at reduced LR
        # ----------------------------------------------------------------
        optimizer_s3 = optim.Adam([params_vec], lr=LEARNING_RATE * 0.3)
 
        for step in range(OPTIM_STEPS_STAGE_3):
            restart_pbar.set_postfix({'Stage': '3/3', 'Step': step + 1})
            optimizer_s3.zero_grad()
            loss = call_jit_model(params_vec, n_samp=50)
            loss.backward()
            optimizer_s3.step()
            clamp_params()
            current_run_history.append(loss.item())
 
        # ---- build model and simulate for this restart ----
        bp          = params_vec.detach().numpy()
        final_model = MouseModelTorch(
            bp[0], bp[1], bp[2], bp[3], bp[4], bp[5], bp[6],
            bp[7], bp[8], bp[9], bp[10], bp[11], bp[12], bp[13],
            fixed_bias_value=fixed_bias.item(),
            initial_boundary=start_phys_bound.item()
        )
        sim_df = simulate_detailed_session_torch(final_model, session_data)
 
        all_restart_results.append({
            'session_id':   session_index,
            'best_params':  bp,
            'fixed_bias':   fixed_bias,
            'nll':          loss.item(),
            'success':      True,
            'sim_df':       sim_df,
            'loss_history': current_run_history,
        })
 
    # ---- pick winner by composite score (NLL + transition asymmetry) ----
    best_result = select_best_restart(all_restart_results, session_data)
    return best_result

def simulate_detailed_session_torch(model, df_session):
    model.reset_state()
    isi_ten, is_short_ten, _ = get_session_tensors(df_session)
 
    # reset the integer index so we can use it safely
    df_session = df_session.reset_index(drop=True)
 
    out = []
    trials_since_switch = 0
    prev_block = None
 
    with torch.no_grad():
        for i, row in df_session.iterrows():
            pre = {
                'p_short_pre':   model.p_short_block.item(),
                'w_time_pre':    model.weights[0].item(),
                'w_ctx_pre':     model.weights[1].item(),
                'bias_pre':      model.last_choice_bias.item(),
                'boundary_pre':  model.current_boundary.item(),
            }
 
            # NEW: track switch position
            curr_block = row['block_type']
            if prev_block is not None and curr_block != prev_block:
                trials_since_switch = 0
            else:
                trials_since_switch += 1
            prev_block = curr_block
 
            isi_t = isi_ten[i]
            p_left, t_perc_avg, _ = model.get_choice_probabilities(
                isi_t, model.current_boundary, n_samples=100
            )
 
            is_left    = torch.rand(1).item() < p_left.item()
            choice_str = 'left' if is_left else 'right'
 
            target_str    = row['trial_type']
            correct_map   = {'short': 'left', 'long': 'right'}
            is_correct    = (choice_str == correct_map[target_str])
            is_short_bool = (target_str == 'short')
 
            # NEW: rare trial flag — trial type disagrees with context belief
            is_rare = (is_short_bool != (pre['p_short_pre'] > 0.5))
 
            bound_shift = 0.0
            if not is_correct:
                bound_shift = model.alpha_boundary.item() if is_short_bool \
                              else -model.alpha_boundary.item()
            model.current_boundary = torch.clamp(
                model.current_boundary + bound_shift, 0.2, 2.3
            )
            model.update_weights(
                torch.tensor(is_correct),
                torch.tensor(is_short_bool),
                t_perc_avg,
                torch.tensor(is_left)
            )
 
            out.append({
                **pre,
                'isi':                 row['isi'],
                'trial_type':          row['trial_type'],
                'block_type':          row['block_type'],
                'mouse_choice':        row['mouse_choice'],
                'model_choice':        choice_str,
                'correct_model':       is_correct,
                'is_rare_trial':       is_rare,          # NEW
                'trials_since_switch': trials_since_switch,  # NEW
            })
 
    return pd.DataFrame(out)

# ------------------------------------------------------------------
# 4.  PLOTTING
# ------------------------------------------------------------------
def logistic_function(x, L, k, x0):
    return L / (1 + np.exp(-k * (x - x0)))

def plot_psych_data_on_ax(ax, df, choice_col, label, color, bins):
    if df.empty:
        return
    
    centers = (bins[:-1] + bins[1:]) / 2
    probs, sems = [], []

    # 1. Calculate Probabilities (P(Right))
    for i in range(len(bins) - 1):
        sub = df[(df['isi'] >= bins[i]) & (df['isi'] < bins[i + 1])]
        if sub.empty:
            probs.append(np.nan)
            sems.append(np.nan)
            continue
        # Calculate P(Right)
        # For Reverse: Short ISI -> Right Lick -> High P(Right)
        p = (sub[choice_col] == 'right').mean()
        n = len(sub)
        probs.append(p)
        sems.append(np.sqrt(p * (1 - p) / n) if n else 0)

    probs = np.array(probs)
    sems = np.array(sems)
    
    # Filter valid data
    ok = ~np.isnan(probs)
    x_data = centers[ok]
    y_data = probs[ok]

    if len(x_data) < 4:
        print(f"Skipping fit for {label}: Not enough data points.")
        return

    # 2. Plot Raw Data Points
    ax.errorbar(x_data, y_data, yerr=sems[ok],
                fmt='o', color=color, label=f'{label} Data',
                capsize=5, markersize=8, alpha=0.7)

    # 3. Robust Curve Fitting (Double-Fit Strategy)
    # We fit TWICE: once assuming normal (k>0), once assuming reverse (k<0).
    # We keep the fit with the lowest Sum of Squared Residuals (SSR).
    
    # Common bounds: L [0.0, 1.0], k [-50, 50], x0 [0.0, 2.5]
    # Note: We relax L bounds to 0.0-1.0 to allow full lapses if needed
    bounds_fit = ([0.0, -50.0, 0.0], [1.0, 50.0, 2.5])
    
    best_popt = None
    best_ssr = float('inf')

    # Try 1: Positive Slope (Standard)
    try:
        p0_pos = [1.0, 5.0, 1.25] # Guess: Max=1, Slope=+5, Bias=1.25
        popt_pos, _ = curve_fit(logistic_function, x_data, y_data, 
                                p0=p0_pos, bounds=bounds_fit, maxfev=5000)
        
        residuals_pos = y_data - logistic_function(x_data, *popt_pos)
        ssr_pos = np.sum(residuals_pos**2)
        
        if ssr_pos < best_ssr:
            best_ssr = ssr_pos
            best_popt = popt_pos
    except:
        pass

    # Try 2: Negative Slope (Reverse)
    try:
        p0_neg = [1.0, -5.0, 1.25] # Guess: Max=1, Slope=-5, Bias=1.25
        popt_neg, _ = curve_fit(logistic_function, x_data, y_data, 
                                p0=p0_neg, bounds=bounds_fit, maxfev=5000)
        
        residuals_neg = y_data - logistic_function(x_data, *popt_neg)
        ssr_neg = np.sum(residuals_neg**2)
        
        if ssr_neg < best_ssr:
            best_ssr = ssr_neg
            best_popt = popt_neg
    except:
        pass

    # 4. Plot the Winner
    if best_popt is not None:
        xfit = np.linspace(bins.min(), bins.max(), 200)
        yfit = logistic_function(xfit, *best_popt)
        ax.plot(xfit, yfit, '-', color=color, linewidth=2, label=f'{label} Fit')
    else:
        print(f"Fit failed for {label}")

def plot_psychometric_comparison(df_exp, df_sim, boundary):
    fig, axs = plt.subplots(1, 3, figsize=(21, 6), sharey=True)
    fig.suptitle('Model vs. Mouse Psychometric Curves (Optimized Beliefs)', fontsize=18)
    bins = np.linspace(0.2, 2.3, 30)

    plot_psych_data_on_ax(axs[0], df_exp, 'mouse_choice', 'Mouse', 'black', bins)
    plot_psych_data_on_ax(axs[0], df_sim, 'model_choice', 'Model', 'red', bins)
    axs[0].set_title('All Blocks')

    plot_psych_data_on_ax(axs[1], df_exp[df_exp['block_type'] == 'short_block'],
                          'mouse_choice', 'Mouse', 'black', bins)
    plot_psych_data_on_ax(axs[1], df_sim[df_sim['block_type'] == 'short_block'],
                          'model_choice', 'Model', 'red', bins)
    axs[1].set_title('Short Block Context')

    plot_psych_data_on_ax(axs[2], df_exp[df_exp['block_type'] == 'long_block'],
                          'mouse_choice', 'Mouse', 'black', bins)
    plot_psych_data_on_ax(axs[2], df_sim[df_sim['block_type'] == 'long_block'],
                          'model_choice', 'Model', 'red', bins)
    axs[2].set_title('Long Block Context')

    for ax in axs:
        ax.set_xlabel('Inter-Stimulus Interval (s)')
        ax.axvline(boundary, color='gray', linestyle=':', label=f'Ref Boundary {boundary:.2f}s')
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
        ax.grid(True, ls='--', alpha=0.6)
        ax.set_ylim(-0.05, 1.05)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axs[0].set_ylabel('P(choose right)')
    handles, labels = axs[0].get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    fig.legend(uniq.values(), uniq.keys(), loc='upper right', bbox_to_anchor=(0.98, 0.85))
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('psychometric_comparison.pdf', dpi=300)
    plt.show()

def plot_internal_states(sim_df):
    sessions_to_plot = sim_df.index.get_level_values('session_id').unique()[:4]
    
    if len(sessions_to_plot) == 0:
        print("Cannot plot internal states: No session data found.")
        return

    fig, axs = plt.subplots(len(sessions_to_plot), 1, 
                            figsize=(15, 3 * len(sessions_to_plot)), 
                            sharex=True, squeeze=False)
    fig.suptitle('Model Internal Belief (p_short_block) vs. Actual Block', fontsize=16)
    
    for i, session_id in enumerate(sessions_to_plot):
        ax = axs[i, 0]
        s_df = sim_df.loc[session_id].reset_index(drop=True)
        
        ax.plot(s_df.index, s_df['p_short_pre'], label='Model p(Short)', color='blue', lw=2)
        ax.set_ylabel('p(Short)')
        ax.set_ylim(-0.05, 1.05)
        ax.axhline(0.5, color='gray', linestyle=':', alpha=0.7)

        block_map = {'short_block': 1, 'long_block': 0, 'neutral': 0.5}
        actual_blocks = s_df['block_type'].map(block_map)
        ax.fill_between(s_df.index, 0, 1, where=(actual_blocks == 1), 
                        color='red', alpha=0.2, label='Actual Short Block')
        ax.fill_between(s_df.index, 0, 1, where=(actual_blocks == 0), 
                        color='green', alpha=0.2, label='Actual Long Block')
        
        ax.set_title(f'Session {session_id}')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        ax.grid(True, ls='--', alpha=0.5)

    ax.set_xlabel('Trial Number')
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])
    plt.savefig('internal_states.pdf', dpi=300)
    plt.show()


def plot_weight_trajectories(sim_df):
    warnings.simplefilter("ignore", category=RuntimeWarning)

    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Model Weight Trajectories', fontsize=16)

    # Individual w_time
    ax = axs[0, 0]
    for session_id in sim_df.index.get_level_values('session_id').unique():
        ax.plot(sim_df.loc[session_id]['w_time_pre'].values, color='gray', alpha=0.2)
    ax.set_title('Individual Session w_time')
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('w_time')

    # Average w_time
    ax = axs[0, 1]
    temp_df = sim_df.reset_index()
    w_time_df = temp_df.pivot(index='trial_in_session', columns='session_id', values='w_time_pre')
    
    mean_w_time = w_time_df.apply(pd.to_numeric, errors='coerce').mean(axis=1)
    sem_w_time = w_time_df.apply(pd.to_numeric, errors='coerce').sem(axis=1)
    ax.plot(mean_w_time.index, mean_w_time, color='blue', lw=2, label='Mean w_time')
    ax.fill_between(mean_w_time.index, mean_w_time - sem_w_time, mean_w_time + sem_w_time,
                    color='blue', alpha=0.2, label='SEM')
    ax.set_title('Average w_time Across Sessions')
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('w_time')
    ax.legend()

    # Individual w_ctx
    ax = axs[1, 0]
    for session_id in sim_df.index.get_level_values('session_id').unique():
        ax.plot(sim_df.loc[session_id]['w_ctx_pre'].values, color='gray', alpha=0.2)
    ax.set_title('Individual Session w_ctx')
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('w_ctx')

    # Average w_ctx
    ax = axs[1, 1]
    w_ctx_df = temp_df.pivot(index='trial_in_session', columns='session_id', values='w_ctx_pre')
    
    mean_w_ctx = w_ctx_df.apply(pd.to_numeric, errors='coerce').mean(axis=1)
    sem_w_ctx = w_ctx_df.apply(pd.to_numeric, errors='coerce').sem(axis=1)
    ax.plot(mean_w_ctx.index, mean_w_ctx, color='green', lw=2, label='Mean w_ctx')
    ax.fill_between(mean_w_ctx.index, mean_w_ctx - sem_w_ctx, mean_w_ctx + sem_w_ctx,
                    color='green', alpha=0.2, label='SEM')
    ax.set_title('Average w_ctx Across Sessions')
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('w_ctx')
    ax.legend()
    
    for ax in axs.flat:
        ax.grid(True, ls='--', alpha=0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    warnings.simplefilter("default", category=RuntimeWarning)
    plt.savefig('weight_trajectories.pdf', dpi=300)
    plt.show()

def plot_all_transition_dynamics(sim_df, exp_df, window=20):
    combined = exp_df.reset_index().set_index(['session_id', 'trial_in_session']).copy()
    cols_to_plot = ['p_short_pre', 'w_time_pre', 'w_ctx_pre', 'bias_pre']
    labels = ['Belief p(Short)', 'Norm. Weight (Time)', 'Norm. Weight (Context)', 'Choice Bias']
    
    sim_aligned = sim_df.reset_index().set_index(['session_id', 'trial_in_session'])[cols_to_plot]
    combined = pd.concat([combined, sim_aligned], axis=1)

    combined['block_numeric'] = combined['block_type'].map({'short_block': 1, 'long_block': 0, 'neutral': np.nan})
    combined['block_diff'] = combined.groupby('session_id')['block_numeric'].diff()
    
    s2l_transitions = combined[combined['block_diff'] == -1].index
    l2s_transitions = combined[combined['block_diff'] == 1].index
    
    data_collector = {col: {'s2l': [], 'l2s': []} for col in cols_to_plot}
    combined_flat = combined.reset_index()

    def normalize_trace(trace):
        t_min, t_max = np.min(trace), np.max(trace)
        if t_max - t_min < 1e-9: return np.zeros_like(trace) 
        return (trace - t_min) / (t_max - t_min)

    def extract_slices(transition_indices, key_type):
        for idx_session, idx_trial in transition_indices:
            matches = combined_flat.index[(combined_flat['session_id'] == idx_session) & 
                                          (combined_flat['trial_in_session'] == idx_trial)]
            if len(matches) == 0: continue
            iloc_pos = matches[0]
            if iloc_pos - window < 0 or iloc_pos + window + 1 > len(combined_flat): continue
            slice_df = combined_flat.iloc[iloc_pos - window : iloc_pos + window + 1]
            if slice_df['session_id'].nunique() == 1 and len(slice_df) == 2 * window + 1:
                for col in cols_to_plot:
                    vals = slice_df[col].values
                    if col in ['w_time_pre', 'w_ctx_pre']:
                        vals = normalize_trace(vals)
                    data_collector[col][key_type].append(vals)

    extract_slices(s2l_transitions, 's2l')
    extract_slices(l2s_transitions, 'l2s')
    
    if not data_collector['p_short_pre']['s2l']:
        print("Not enough transitions found to plot dynamics.")
        return

    x = np.arange(-window, window + 1)
    fig, axs = plt.subplots(len(cols_to_plot), 2, figsize=(14, 3.5 * len(cols_to_plot)), sharex=True)
    fig.suptitle('Model Dynamics Around Block Transitions (Weights Normalized per Window)', fontsize=18)
    
    warnings.simplefilter("ignore", category=RuntimeWarning)

    for i, (col, label) in enumerate(zip(cols_to_plot, labels)):
        ax_left = axs[i, 0]
        arr_s2l = np.array(data_collector[col]['s2l'])
        mean_s2l = np.nanmean(arr_s2l, axis=0)
        sem_s2l = np.nanstd(arr_s2l, axis=0) / np.sqrt(np.sum(~np.isnan(arr_s2l), axis=0))
        
        ax_left.plot(x, mean_s2l, color='green', lw=2)
        ax_left.fill_between(x, mean_s2l - sem_s2l, mean_s2l + sem_s2l, color='green', alpha=0.2)
        ax_left.axvline(0, color='black', linestyle='--', alpha=0.5)
        ax_left.set_ylabel(label, fontsize=12)
        if i == 0: ax_left.set_title(f'Short -> Long (n={len(arr_s2l)})', fontsize=14)
        
        ax_right = axs[i, 1]
        arr_l2s = np.array(data_collector[col]['l2s'])
        mean_l2s = np.nanmean(arr_l2s, axis=0)
        sem_l2s = np.nanstd(arr_l2s, axis=0) / np.sqrt(np.sum(~np.isnan(arr_l2s), axis=0))
        
        ax_right.plot(x, mean_l2s, color='red', lw=2)
        ax_right.fill_between(x, mean_l2s - sem_l2s, mean_l2s + sem_l2s, color='red', alpha=0.2)
        ax_right.axvline(0, color='black', linestyle='--', alpha=0.5)
        if i == 0: ax_right.set_title(f'Long -> Short (n={len(arr_l2s)})', fontsize=14)

        for ax in [ax_left, ax_right]:
            ax.grid(True, ls=':', alpha=0.6)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            if col in ['w_time_pre', 'w_ctx_pre']:
                ax.set_ylim(-0.05, 1.05) 
            if i == len(cols_to_plot) - 1:
                ax.set_xlabel('Trials from Switch', fontsize=12)

    warnings.simplefilter("default", category=RuntimeWarning)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('transition_dynamics.pdf', dpi=300)
    plt.show()


def plot_rare_trial_dynamics(sim_df, window=2):
    df = sim_df.reset_index().copy()
    cols_to_plot = ['p_short_pre', 'w_time_pre', 'w_ctx_pre', 'bias_pre']
    labels = ['Belief p(Short)', 'Norm. Weight (Time)', 'Norm. Weight (Context)', 'Choice Bias']
    
    is_short_blk = df['block_type'] == 'short_block'
    is_long_blk  = df['block_type'] == 'long_block'
    is_short_tr  = df['trial_type'] == 'short'
    is_long_tr   = df['trial_type'] == 'long'
    
    idx_rare_long = df[is_short_blk & is_long_tr].index
    idx_rare_short = df[is_long_blk & is_short_tr].index
    
    data_collector = {col: {'rare_long': [], 'rare_short': []} for col in cols_to_plot}

    def normalize_trace(trace):
        t_min, t_max = np.min(trace), np.max(trace)
        if t_max - t_min < 1e-9: return np.zeros_like(trace)
        return (trace - t_min) / (t_max - t_min)

    def collect_windows(indices, key_type):
        for idx in indices:
            if idx - window < 0 or idx + window + 1 > len(df): continue
            slice_df = df.iloc[idx - window : idx + window + 1]
            if slice_df['session_id'].nunique() == 1:
                for col in cols_to_plot:
                    vals = slice_df[col].values
                    if col in ['w_time_pre', 'w_ctx_pre']:
                        vals = normalize_trace(vals)
                    data_collector[col][key_type].append(vals)

    collect_windows(idx_rare_long, 'rare_long')
    collect_windows(idx_rare_short, 'rare_short')

    if not data_collector['p_short_pre']['rare_long'] and not data_collector['p_short_pre']['rare_short']:
        print("No rare trials found suitable for plotting.")
        return

    x = np.arange(-window, window + 1)
    fig, axs = plt.subplots(len(cols_to_plot), 2, figsize=(12, 3 * len(cols_to_plot)), sharex=True)
    fig.suptitle(f'Model Dynamics Around Rare Trials (Weights Normalized, Win +/-{window})', fontsize=16)
    
    warnings.simplefilter("ignore", category=RuntimeWarning)

    for i, (col, label) in enumerate(zip(cols_to_plot, labels)):
        ax1 = axs[i, 0]
        vals = np.array(data_collector[col]['rare_long'])
        if len(vals) > 0:
            mean_v = np.nanmean(vals, axis=0)
            sem_v = np.nanstd(vals, axis=0) / np.sqrt(np.sum(~np.isnan(vals), axis=0))
            ax1.plot(x, mean_v, color='purple', lw=2, marker='o', markersize=4)
            ax1.fill_between(x, mean_v - sem_v, mean_v + sem_v, color='purple', alpha=0.2)
        
        ax1.axvline(0, color='gray', linestyle='--', alpha=0.5)
        ax1.set_ylabel(label, fontsize=11)
        if i == 0: ax1.set_title(f'Rare LONG in Short Block', fontsize=12, color='purple')

        ax2 = axs[i, 1]
        vals = np.array(data_collector[col]['rare_short'])
        if len(vals) > 0:
            mean_v = np.nanmean(vals, axis=0)
            sem_v = np.nanstd(vals, axis=0) / np.sqrt(np.sum(~np.isnan(vals), axis=0))
            ax2.plot(x, mean_v, color='orange', lw=2, marker='o', markersize=4)
            ax2.fill_between(x, mean_v - sem_v, mean_v + sem_v, color='orange', alpha=0.2)
            
        ax2.axvline(0, color='gray', linestyle='--', alpha=0.5)
        if i == 0: ax2.set_title(f'Rare SHORT in Long Block', fontsize=12, color='orange')

        for ax in [ax1, ax2]:
            ax.set_xticks(x)
            ax.grid(True, ls=':', alpha=0.6)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            if col in ['w_time_pre', 'w_ctx_pre']:
                ax.set_ylim(-0.05, 1.05)
            if i == len(cols_to_plot) - 1:
                ax.set_xlabel('Trials relative to Rare Event')

    warnings.simplefilter("default", category=RuntimeWarning)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('rare_trial_dynamics.pdf', dpi=300)
    plt.show()

def plot_performance_around_transitions(exp_df, sim_df, window=15):
    combined = exp_df.reset_index().set_index(['session_id', 'trial_in_session']).copy()
    sim_aligned = sim_df.reset_index().set_index(['session_id', 'trial_in_session'])
    
    correct_map = {'short': 'left', 'long': 'right'}
    combined['correct_mouse'] = combined.apply(
        lambda row: 1 if row['mouse_choice'] == correct_map[row['trial_type']] else 0, axis=1
    )
    combined['correct_model'] = sim_aligned['correct_model'].astype(int)
    
    combined['block_numeric'] = combined['block_type'].map({'short_block': 1, 'long_block': 0, 'neutral': np.nan})
    combined['block_diff'] = combined.groupby('session_id')['block_numeric'].diff()
    
    s2l_trans = combined[combined['block_diff'] == -1].index
    l2s_trans = combined[combined['block_diff'] == 1].index
    
    combined_flat = combined.reset_index()
    
    def get_event_traces(indices):
        mouse_traces = []
        model_traces = []
        for idx_sess, idx_trial in indices:
            matches = combined_flat.index[(combined_flat['session_id'] == idx_sess) & 
                                          (combined_flat['trial_in_session'] == idx_trial)]
            if len(matches) == 0: continue
            loc = matches[0]
            
            if loc - window < 0 or loc + window + 1 > len(combined_flat): continue
            slice_df = combined_flat.iloc[loc - window : loc + window + 1]
            if slice_df['session_id'].nunique() == 1:
                mouse_traces.append(slice_df['correct_mouse'].values)
                model_traces.append(slice_df['correct_model'].values)
        return mouse_traces, model_traces

    s2l_mouse, s2l_model = get_event_traces(s2l_trans)
    l2s_mouse, l2s_model = get_event_traces(l2s_trans)
    
    if not s2l_mouse or not l2s_mouse:
        print("Not enough transitions for performance plot.")
        return

    fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    fig.suptitle(f'Performance (Accuracy) Around Block Transitions (Window +/-{window})', fontsize=16)
    
    x = np.arange(-window, window + 1)
    
    def plot_trace(ax, data_list, color, label):
        arr = np.array(data_list)
        mean = np.nanmean(arr, axis=0)
        sem = np.nanstd(arr, axis=0) / np.sqrt(np.sum(~np.isnan(arr), axis=0))
        ax.plot(x, mean, color=color, lw=1.5, label=label)
        ax.fill_between(x, mean - sem, mean + sem, color=color, alpha=0.15)

    plot_trace(axs[0], s2l_mouse, 'black', 'Mouse')
    plot_trace(axs[0], s2l_model, 'red', 'Model')
    axs[0].set_title(f'Short -> Long Switch (n={len(s2l_mouse)})')
    axs[0].set_ylabel('Proportion Rewarded')
    
    plot_trace(axs[1], l2s_mouse, 'black', 'Mouse')
    plot_trace(axs[1], l2s_model, 'red', 'Model')
    axs[1].set_title(f'Long -> Short Switch (n={len(l2s_mouse)})')
    
    for ax in axs:
        ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
        ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance')
        ax.set_xlabel('Trials from Switch')
        ax.set_ylim(0, 1.05)
        ax.legend(loc='lower right')
        ax.grid(True, ls=':', alpha=0.5)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('performance_around_transitions.pdf', dpi=300)
    plt.show()


def plot_performance_around_rare_trials(exp_df, sim_df, window=3):
    combined = exp_df.reset_index().set_index(['session_id', 'trial_in_session']).copy()
    sim_aligned = sim_df.reset_index().set_index(['session_id', 'trial_in_session'])
    
    correct_map = {'short': 'left', 'long': 'right'}
    combined['correct_mouse'] = combined.apply(
        lambda row: 1 if row['mouse_choice'] == correct_map[row['trial_type']] else 0, axis=1
    )
    combined['correct_model'] = sim_aligned['correct_model'].astype(int)
    
    is_short_blk = combined['block_type'] == 'short_block'
    is_long_blk  = combined['block_type'] == 'long_block'
    is_short_tr  = combined['trial_type'] == 'short'
    is_long_tr   = combined['trial_type'] == 'long'
    
    idx_rare_long = combined[is_short_blk & is_long_tr].index
    idx_rare_short = combined[is_long_blk & is_short_tr].index
    
    combined_flat = combined.reset_index()
    
    def get_event_traces(indices):
        mouse_traces = []
        model_traces = []
        for idx_sess, idx_trial in indices:
            matches = combined_flat.index[(combined_flat['session_id'] == idx_sess) & 
                                          (combined_flat['trial_in_session'] == idx_trial)]
            if len(matches) == 0: continue
            loc = matches[0]
            if loc - window < 0 or loc + window + 1 > len(combined_flat): continue
            slice_df = combined_flat.iloc[loc - window : loc + window + 1]
            if slice_df['session_id'].nunique() == 1:
                mouse_traces.append(slice_df['correct_mouse'].values)
                model_traces.append(slice_df['correct_model'].values)
        return mouse_traces, model_traces

    rl_mouse, rl_model = get_event_traces(idx_rare_long)
    rs_mouse, rs_model = get_event_traces(idx_rare_short)

    if not rl_mouse and not rs_mouse:
        print("Not enough rare trials for performance plot.")
        return

    fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    fig.suptitle(f'Performance Around Rare Trials (Window +/-{window})', fontsize=16)
    x = np.arange(-window, window + 1)

    def plot_trace(ax, data_list, color, label):
        if not data_list: return
        arr = np.array(data_list)
        mean = np.nanmean(arr, axis=0)
        sem = np.nanstd(arr, axis=0) / np.sqrt(np.sum(~np.isnan(arr), axis=0))
        ax.plot(x, mean, color=color, lw=1.5, marker='o', markersize=4, label=label)
        ax.fill_between(x, mean - sem, mean + sem, color=color, alpha=0.15)

    plot_trace(axs[0], rl_mouse, 'black', 'Mouse')
    plot_trace(axs[0], rl_model, 'red', 'Model')
    axs[0].set_title(f'Rare LONG (in Short Block)\n(n={len(rl_mouse)})')
    axs[0].set_ylabel('Proportion Correct')

    plot_trace(axs[1], rs_mouse, 'black', 'Mouse')
    plot_trace(axs[1], rs_model, 'red', 'Model')
    axs[1].set_title(f'Rare SHORT (in Long Block)\n(n={len(rs_mouse)})')

    for ax in axs:
        ax.set_xticks(x)
        ax.axvline(0, color='gray', linestyle='--', alpha=0.7, label='Rare Trial')
        ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
        ax.set_xlabel('Trials relative to Rare Event')
        ax.set_ylim(0, 1.05)
        ax.legend()
        ax.grid(True, ls=':', alpha=0.5)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('performance_around_rare_trials.pdf', dpi=300)
    plt.show()

def plot_boundary_dynamics(sim_df):
    sessions_to_plot = sim_df.index.get_level_values('session_id').unique()[:4]
    if len(sessions_to_plot) == 0: return

    fig, axs = plt.subplots(len(sessions_to_plot), 1, figsize=(15, 3 * len(sessions_to_plot)), sharex=True)
    fig.suptitle('Dynamic Decision Boundary Evolution', fontsize=16)

    # Handle single session case to ensure axs is iterable
    if len(sessions_to_plot) == 1:
        axs = [axs]

    for i, session_id in enumerate(sessions_to_plot):
        ax = axs[i]
        s_df = sim_df.loc[session_id].reset_index(drop=True)
        
        # 1. Plot Boundary Line
        ax.plot(s_df.index, s_df['boundary_pre'], color='purple', lw=2, label='Boundary')
        ax.axhline(1.25, color='gray', linestyle=':', label='Start (1.25s)')
        
        # 2. Add Block Background Shading
        # We define a range that covers the possible boundary values (0.5 to 2.5)
        y_min, y_max = 0.0, 3.0 
        
        is_short = (s_df['block_type'] == 'short_block')
        is_long = (s_df['block_type'] == 'long_block')
        
        ax.fill_between(s_df.index, y_min, y_max, where=is_short, 
                        color='red', alpha=0.1, label='Short Block')
        ax.fill_between(s_df.index, y_min, y_max, where=is_long, 
                        color='green', alpha=0.1, label='Long Block')
        
        # 3. Highlight Errors
        errors = s_df[~s_df['correct_model']]
        ax.scatter(errors.index, errors['boundary_pre'], color='red', s=10, alpha=0.6, label='Error', zorder=5)

        # 4. Formatting
        ax.set_ylabel('Boundary (s)')
        ax.set_title(f'Session {session_id}')
        ax.set_ylim(0.1, 2.5) # Focus on the active range
        ax.grid(True, alpha=0.3)
        
        # Clean Legend (avoid duplicates)
        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            ax.legend(by_label.values(), by_label.keys(), loc='upper right')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('boundary_dynamics.pdf')
    plt.show()

def plot_loss_evolution(results):
    """
    Plots optimization loss separated by stage for each session.
    Creates a (N_sessions x 3) grid.
    """
    valid_results = [r for r in results if r['success']]
    if not valid_results:
        print("No valid results to plot loss.")
        return

    n_sessions = len(valid_results)
    
    # Create a grid: Rows = Sessions, Cols = 3 Stages
    # Adjust height: 3 inches per session
    fig, axs = plt.subplots(n_sessions, 3, figsize=(15, 3 * n_sessions), constrained_layout=True)
    
    # If only 1 session, axs is 1D array. Make it 2D (1, 3) for consistency.
    if n_sessions == 1:
        axs = np.expand_dims(axs, axis=0)

    # Define indices for slicing
    idx_s1_end = OPTIM_STEPS_STAGE_1
    idx_s2_end = OPTIM_STEPS_STAGE_1 + OPTIM_STEPS_STAGE_2
    
    for i, res in enumerate(valid_results):
        loss = np.array(res['loss_history'])
        sess_id = res['session_id']
        
        # Slice the data
        loss_s1 = loss[0 : idx_s1_end]
        loss_s2 = loss[idx_s1_end : idx_s2_end]
        loss_s3 = loss[idx_s2_end :]
        
        # --- Plot Stage 1: Sensory ---
        ax1 = axs[i, 0]
        ax1.plot(range(len(loss_s1)), loss_s1, color='blue')
        ax1.set_title(f'Sess {sess_id}: Stage 1 (Sensory)')
        ax1.set_ylabel('NLL')
        ax1.grid(True, alpha=0.3)
        
        # --- Plot Stage 2: Strategy ---
        ax2 = axs[i, 1]
        # X-axis continues from where S1 left off for clarity, or starts at 0. 
        # Let's start at 0 for "Step in Stage"
        ax2.plot(range(len(loss_s2)), loss_s2, color='green')
        ax2.set_title(f'Sess {sess_id}: Stage 2 (Strategy)')
        # Auto-scale Y to see small changes
        ax2.autoscale(enable=True, axis='y', tight=False)
        ax2.grid(True, alpha=0.3)

        # --- Plot Stage 3: Fine Tune ---
        ax3 = axs[i, 2]
        ax3.plot(range(len(loss_s3)), loss_s3, color='black')
        ax3.set_title(f'Sess {sess_id}: Stage 3 (FineTune)')
        ax3.autoscale(enable=True, axis='y', tight=False)
        ax3.grid(True, alpha=0.3)

    plt.suptitle('Optimization Progress per Stage (Independent Y-Scales)', fontsize=16)
    plt.savefig('optimization_loss_separated.pdf')
    plt.show()

# ------------------------------------------------------------------
# 5.  SAVE COMPREHENSIVE RESULTS 
# ------------------------------------------------------------------

def save_comprehensive_results(results, prefix='Model_Fit'):
    """
    Saves two files:
    1. prefix_Trials_and_Params.csv: Contains ALL trial data merged with best fitted parameters.
    2. prefix_Loss_History.csv: Contains the step-by-step loss evolution for every session.
    """
    if not results:
        print("No results to save.")
        return

    # --- 1. Define Parameter Names ---
    # Must match the order in fit_single_session_torch
    param_names = [
        'decay_rate', 'noise_param_a', 
        'alpha_reward', 'alpha_punish', 'alpha_unc',
        'alpha_ch_r', 'alpha_ch_p', 
        'beta', 'lapse',
        'p_switch', 'p_rare', 
        'alpha_boundary' 
    ]

    all_sessions_dfs = []
    loss_data = {}

    print(f"\nCompiling data for export...")
    
    for res in results:
        if not res['success']: 
            continue
            
        sess_id = res['session_id']
        params = res['best_params']
        fixed_bias = res['fixed_bias']
        if torch.is_tensor(fixed_bias): fixed_bias = fixed_bias.item()
        
        final_nll = res['nll']
        sim_df = res['sim_df'].copy()
        
        # --- A. Attach Session Metadata & Parameters to DataFrame ---
        # This repeats the parameter values for every trial in this session,
        # which makes the CSV self-contained for analysis.
        sim_df.insert(0, 'session_id', sess_id)
        sim_df['final_nll'] = final_nll
        sim_df['fixed_bias_param'] = fixed_bias
        
        for i, p_name in enumerate(param_names):
            sim_df[p_name] = params[i]
            
        all_sessions_dfs.append(sim_df)
        
        # --- B. Collect Loss History ---
        # We assume loss_history exists from our previous update
        if 'loss_history' in res:
            loss_data[f'Session_{sess_id}'] = res['loss_history']

    # --- SAVE FILE 1: TRIAL DATA + PARAMS ---
    if all_sessions_dfs:
        full_df = pd.concat(all_sessions_dfs, ignore_index=True)
        
        # Rename columns to be nice and readable if needed
        # (sim_df already has: isi, trial_type, block_type, mouse_choice, etc.)
        
        csv_name = f"{prefix}_Trials_and_Params.csv"
        full_df.to_csv(csv_name, index=False)
        print(f"✔ Saved comprehensive trial data to: {csv_name}")
        print(f"  (Shape: {full_df.shape})")

    # --- SAVE FILE 2: LOSS HISTORY ---
    if loss_data:
        # Pad lists with NaN if they have different lengths (though they shouldn't)
        max_len = max(len(v) for v in loss_data.values())
        padded_data = {k: v + [np.nan]*(max_len - len(v)) for k, v in loss_data.items()}
        
        loss_df = pd.DataFrame(padded_data)
        loss_df.index.name = 'Step'
        
        loss_name = f"{prefix}_Loss_History.csv"
        loss_df.to_csv(loss_name)
        print(f"✔ Saved optimization history to: {loss_name}")

# ------------------------------------------------------------------
# 5.  EXECUTION BLOCK
# ------------------------------------------------------------------
if __name__ == '__main__':
    # REPLACE WITH YOUR ACTUAL PATH
    DATA_PATHS = [
    # r'/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/Behavior/YH24LG/YH24LG_block_single_interval_discrimination_V_1_20250814_121831.mat',
    # r"/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/Behavior/SCHR_MC06/MC06_SChR2_block_single_interval_discrimination_V_1_20250721_151425.mat",
    r"/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/Behavior/SCHR_MC06/MC06_SChR2_block_single_interval_discrimination_V_1_20250801_144807.mat",
    # [Uncomment other paths as needed]
    ]
    
    if len(DATA_PATHS) == 0:
        print("Please populate DATA_PATHS in the script.")
    else:
        print("Loading Data...")
        raw = prepare_session_data(DATA_PATHS)
        if not raw['outcomes']: 
            print("No outcomes loaded. Check paths.")
            sys.exit()
        
        sessions_df = [transform_data_to_dataframe(raw, i) for i in range(len(raw['outcomes']))]
        sessions_df = [df for df in sessions_df if not df.empty]
        print(f'Loaded {len(sessions_df)} sessions')

        # ALL 14 PARAMETERS
        bounds = [
            (0.2, 1.2),      # 0: decay (Internal clock speed)
            (1.0, 60.0),     # 1: noise (Sensory precision)
            (0.0, 0.2),      # 2: a_rew (Safe base learning rate)
            (0.0, 0.2),      # 3: a_pun (Safe base learning rate)
            (0.0, 1.0),      # 4: gamma (Balance Dial)
            (0.0, 2.5),      # 5: a_unc_ctx (Limits explosive weight shifts)
            (0.0, 2.5),      # 6: a_unc_sens (Limits explosive weight shifts)
            (0.0, 1.0),      # 7: a_ch_r (Choice history reward)
            (0.0, 1.0),      # 8: a_ch_p (Choice history punish)
            (0.1, 15.0),     # 9: beta (Softmax inverse temp)
            (0.0, 0.3),      # 10: lapse (Capped at 30% to prevent lazy fitting)
            (0.001, 0.25),   # 11: p_switch (Mouse expects switch every 4 to 1000 trials)
            (0.01, 0.45),    # 12: p_rare (Prevents mathematical collapse to 50/50)
            (0.0, 0.2)       # 13: alpha_boundary (Max physical boundary shift)
        ]

        n_cpus = max(1, multiprocessing.cpu_count() - 1)
        print(f"Starting 3-Stage Fit with JIT Optimization ... ({n_cpus} cores)")
        
        pool = multiprocessing.Pool(processes=n_cpus)
        fit_tasks = [(df, i, bounds, None) for i, df in enumerate(sessions_df)]
        
        results = list(tqdm(pool.imap(fit_single_session_torch, fit_tasks), total=len(sessions_df), desc="Sessions"))
        pool.close()
        
        sim_dfs_list = [r['sim_df'] for r in results]
        valid_params = [r['best_params'] for r in results if r['success']]
        fixed_biases = [r['fixed_bias'] for r in results]
        
        param_names_reduced = [
            'decay_rate', 'noise_param_a', 'alpha_reward', 'alpha_punish', 
            'gamma', 'alpha_unc_ctx', 'alpha_unc_sens',
            'alpha_ch_r', 'alpha_ch_p', 'beta', 'lapse',
            'p_switch', 'p_rare', 'alpha_boundary'
        ]

        if valid_params:
            mean_params = np.mean(valid_params, axis=0)
            print(f'\nMEAN PARAMETERS (3 Stages):')
            for n, v in zip(param_names_reduced, mean_params):
                print(f'   {n}: {v:.4f}')
            print(f'   Average Fixed Bias: {np.mean(fixed_biases):.4f} +- {np.std(fixed_biases):.4f}')
        
        if sessions_df:
            exp_all = pd.concat(sessions_df, keys=range(len(sessions_df)), names=['session_id', 'trial_in_session']).reset_index(drop=True)
            sim_all = pd.concat(sim_dfs_list, keys=range(len(sim_dfs_list)), names=['session_id', 'trial_in_session']).reset_index(drop=True)
            
            correct_map = {'short': 'left', 'long': 'right'}
            comparison = sim_all.copy()
            comparison['correct_mouse'] = exp_all.apply(
                lambda row: 1 if row['mouse_choice'] == correct_map[row['trial_type']] else 0, axis=1
            )
            comparison['correct_model'] = comparison['correct_model'].astype(int)
            comparison['block_type'] = exp_all['block_type']
            comparison['isi'] = exp_all['isi']

            mouse_acc = comparison['correct_mouse'].mean()
            model_acc = comparison['correct_model'].mean()
            print(f"\nACCURACY COMPARISON")
            print(f"   Mouse overall:  {mouse_acc:.3%}")
            print(f"   Model overall:  {model_acc:.3%}")
            print(f"   Difference:     {model_acc - mouse_acc:+.3%}")

            print(f"\nBY BLOCK TYPE:")
            for block in ['short_block', 'long_block', 'neutral']:
                subset = comparison[comparison['block_type'] == block]
                if len(subset) == 0: continue
                print(f"   {block:12}: Mouse {subset['correct_mouse'].mean():.3%} | Model {subset['correct_model'].mean():.3%}")

            print(f"\nBY ISI BIN:")
            isi_bins = pd.cut(comparison['isi'], bins=np.linspace(0, 2.5, 21))
            acc_by_bin = comparison.groupby(isi_bins, observed=False).agg(
                mouse_acc=('correct_mouse', 'mean'),
                model_acc=('correct_model', 'mean'),
                n=('correct_mouse', 'size')
            ).round(3)
            print(acc_by_bin)

            agreement = (comparison['mouse_choice'] == comparison['model_choice']).mean()
            print(f"\nTRIAL-BY-TRIAL CHOICE AGREEMENT: {agreement:.3%}")
            
            # --- PLOTTING ---
            exp_all_with_keys = pd.concat(sessions_df, keys=range(len(sessions_df)), names=['session_id', 'trial_in_session'])
            sim_all_with_keys = pd.concat(sim_dfs_list, keys=range(len(sim_dfs_list)), names=['session_id', 'trial_in_session'])

            print("\nGenerating Plots...")
            plot_psychometric_comparison(exp_all, sim_all, boundary=1.25)
            plot_internal_states(sim_all_with_keys)
            plot_loss_evolution(results)
            plot_weight_trajectories(sim_all_with_keys)
            plot_boundary_dynamics(sim_all_with_keys)
            plot_all_transition_dynamics(sim_all_with_keys, exp_all_with_keys)
            plot_rare_trial_dynamics(sim_all_with_keys, window=2)
            plot_performance_around_transitions(exp_all_with_keys, sim_all_with_keys, window=10)
            plot_performance_around_rare_trials(exp_all_with_keys, sim_all_with_keys, window=3)

            # --- NEW: SAVE EVERYTHING ---
            save_comprehensive_results(results, prefix='Dynamic_Boundary_Fit')

Loading Data...
Loaded 1 sessions
Starting 3-Stage Fit with JIT Optimization ... (39 cores)


Sess 0 Restarts:   0%|          | 0/5 [00:00<?, ?it/s]